# 04 - ORION Latent Causal Stability

**Question:** Does a downstream decoder depend on the same latent time windows in two contexts?

This toy example compares two continuous ORION representations. In real work, contexts might be sessions, subjects, datasets, tokenizers, or model checkpoints.

**Evidence tier:** scientific synthetic. A real cross-session claim requires held-out real data.


In [ ]:
import numpy as np
from orion.contracts import RepresentationBatch
from neuros_mechint import EvidenceTier
from neuros_mechint.benchmarks import compare_effect_maps, extract_effect_map
from neuros_mechint.integrations.orion import representation_window_audit


In [ ]:
timestamps = np.arange(6, dtype=np.int64) * 10_000_000
session_a = RepresentationBatch(
    values=np.array([[1, 0], [2, 0], [8, 2], [9, 2], [2, 0], [1, 0]], dtype=np.float32),
    timestamps_ns=timestamps,
)
session_b = RepresentationBatch(
    values=np.array([[1, 0], [2, 0], [7, 2], [8, 2], [2, 0], [1, 0]], dtype=np.float32),
    timestamps_ns=timestamps,
)

def scorer(batch):
    return float(np.asarray(batch.values)[:, 0].sum())

audit_a = representation_window_audit(
    session_a, scorer, window_ns=20_000_000,
    evidence_tier=EvidenceTier.SCIENTIFIC_SYNTHETIC, seed=1,
)
audit_b = representation_window_audit(
    session_b, scorer, window_ns=20_000_000,
    evidence_tier=EvidenceTier.SCIENTIFIC_SYNTHETIC, seed=1,
)

stability = compare_effect_maps(
    extract_effect_map(audit_a),
    extract_effect_map(audit_b),
    top_k=2,
)
stability.to_dict()


## Interpret cautiously

High correlation alone is not enough. Inspect sign agreement, top-k overlap, the number of shared intervention targets, and absolute effect drift. Then repeat on held-out contexts.

For a stronger cross-session experiment, keep window definitions aligned in physical time or event-relative coordinates, hold the downstream metric fixed, and compare against matched random-window baselines.
